# EM Displacement VLM — Colab bootstrap

Lightweight setup (clone, Drive, install). For **A100 fine-tuning**, use [`colab_a100.ipynb`](https://colab.research.google.com/github/rlogger/em-displacement-vlm/blob/main/notebooks/colab_a100.ipynb) instead.

Runtime → GPU → **A100** recommended.

In [ ]:
!nvidia-smi
import torch
print("cuda:", torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else None)

## 1. Drive mount

In [ ]:
from pathlib import Path
import os

MOUNT_DRIVE = True
DRIVE_PROJECT = Path("/content/drive/MyDrive/em-displacement-vlm")

if MOUNT_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive", force_remount=False)
    for sub in ("data", "checkpoints", "results"):
        (DRIVE_PROJECT / sub).mkdir(parents=True, exist_ok=True)
    os.environ["EM_DATA_DIR"] = str(DRIVE_PROJECT / "data")
    os.environ["EM_CHECKPOINT_DIR"] = str(DRIVE_PROJECT / "checkpoints")
    os.environ["EM_RESULTS_DIR"] = str(DRIVE_PROJECT / "results")
    print("Drive project:", DRIVE_PROJECT)
else:
    print("Drive mount skipped — using /content for ephemeral storage.")

## 2. Clone / pull

In [ ]:
from pathlib import Path

REPO_URL = "https://github.com/rlogger/em-displacement-vlm.git"
REPO_DIR = Path("/content/em-displacement-vlm")
BRANCH = "main"

if REPO_DIR.exists() and (REPO_DIR / ".git").exists():
    %cd {REPO_DIR}
    !git fetch origin
    !git checkout {BRANCH}
    !git pull --ff-only origin {BRANCH}
else:
    !git clone --branch {BRANCH} {REPO_URL} {REPO_DIR}
    %cd {REPO_DIR}

!git rev-parse --short HEAD
!git status -sb

## 3. Install

In [ ]:
%pip install -q -e ".[vlm,dev]"

from em_displacement_vlm.runtime import runtime_info
from em_displacement_vlm.paths import data_dir, checkpoint_dir

for k, v in runtime_info().items():
    print(f"{k}: {v}")
print("data_dir:", data_dir())
print("checkpoint_dir:", checkpoint_dir())
print("\nFor Gemma FT on A100 open: notebooks/colab_a100.ipynb")

## 4. Secrets

In [ ]:
from google.colab import userdata
import os

def _set_secret(name: str) -> None:
    try:
        os.environ[name] = userdata.get(name)
        print(f"Loaded secret: {name}")
    except Exception:
        print(f"Secret not set (ok if unused): {name}")

for key in ("HF_TOKEN", "WANDB_API_KEY", "GITHUB_TOKEN"):
    _set_secret(key)

if os.environ.get("HF_TOKEN"):
    !huggingface-cli login --token "$HF_TOKEN" --add-to-git-credential